# Complete Training Script for Art Images Dataset
### Dataset: Art Images - Drawing/Painting/Sculptures/Engravings
Train Attention GAN to generate artistic image

In [1]:
# import torch
import os
import sys
from PIL import Image
from tqdm import tqdm
import json

# Add project paths
sys.path.append('./models')
sys.path.append('./utils')

In [ ]:
# DATASET PREPARATION

print("\n" + "=" * 80)
print("PREPARING ART IMAGES DATASET")
print("=" * 80)

DATA_DIR = './data/art_images'  

# Check if dataset exists
if not os.path.exists(DATA_DIR):
    print(f"\n Dataset not found at {DATA_DIR}")
    print("\nPlease download the dataset first:")
    print("1. Go to Kaggle: https://www.kaggle.com/datasets/thedownhill/art-images-drawings-painting-sculpture-engraving")
    print("2. Download the dataset")
    print("3. Extract to './data/art_images'")
    print("\nExpected structure:")
    print("./data/art_images/")
    print("  ├── drawings/")
    print("  ├── engraving/")
    print("  ├── painting/")
    print("  └── sculpture/")
    sys.exit(1)

# Find all image files in subdirectories
print(f"\n Scanning {DATA_DIR} for images...")

all_images = []
categories = ['drawings', 'engraving', 'painting', 'sculpture']

for category in categories:
    category_path = os.path.join(DATA_DIR, category)
    if os.path.exists(category_path):
        images = [
            os.path.join(category, f)
            for f in os.listdir(category_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        all_images.extend(images)
        print(f"  ✓ {category}: {len(images)} images")

print(f"\n Total images found: {len(all_images)}")

if len(all_images) == 0:
    print(" No images found! Please check dataset structure.")
    sys.exit(1)

In [ ]:
# STEP 2: GENERATE CAPTIONS 

print("\n" + "=" * 80)
print("STEP 2: CHECKING/GENERATING CAPTIONS")
print("=" * 80)

# Check if captions already exist
captions_exist = False
sample_image = all_images[0]
sample_txt = os.path.join(DATA_DIR, sample_image.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt'))
captions_exist = os.path.exists(sample_txt)

if captions_exist:
    print("\n Captions already exist! Skipping caption generation.")
else:
    print("\n Captions not found. Generating captions automatically...")
    print("This may take a while depending on dataset size...")

    response = input("\nGenerate captions now? (y/n): ").strip().lower()

    if response == 'y':
        from caption_generator import AutoCaptioner

        # Initialize captioner
        print("\n Initializing BLIP-2 captioner...")
        captioner = AutoCaptioner(model_type='blip2')

        # Generate captions for each category
        for category in categories:
            category_path = os.path.join(DATA_DIR, category)
            if os.path.exists(category_path):
                print(f"\n Generating captions for {category}...")

                # Add art style to captions
                style_prefix = {
                    'drawings': 'a drawing of',
                    'engraving': 'an engraving of',
                    'painting': 'a painting of',
                    'sculpture': 'a sculpture of'
                }

                images = [f for f in os.listdir(category_path)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

                for img_file in tqdm(images, desc=f"Captioning {category}"):
                    img_path = os.path.join(category_path, img_file)

                    # Generate caption
                    try:
                        caption = captioner.caption_image(img_path)

                        # Add style information
                        enhanced_caption = f"{style_prefix[category]} {caption}, art, artwork, artistic"

                        # Save caption
                        txt_path = img_path.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
                        with open(txt_path, 'w', encoding='utf-8') as f:
                            f.write(enhanced_caption)
                    except Exception as e:
                        print(f"Error with {img_file}: {e}")

        print("\n Caption generation complete!")
    else:
        print("\n  Skipping caption generation. Using image filenames as captions.")
        print("Note: This will result in lower quality. Consider generating captions later.")


In [ ]:
# ORGANIZE DATASET

print("\n" + "=" * 80)
print("STEP 3: ORGANIZING DATASET")
print("=" * 80)

# Create organized structure
ORGANIZED_DIR = './data/art_images_organized'
os.makedirs(f'{ORGANIZED_DIR}/train', exist_ok=True)
os.makedirs(f'{ORGANIZED_DIR}/val', exist_ok=True)

print(f"\n Organizing images into train/val split...")

import random
random.shuffle(all_images)

# 90% train, 10% validation
split_idx = int(len(all_images) * 0.9)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

print(f"  Training set: {len(train_images)} images")
print(f"  Validation set: {len(val_images)} images")

# Copy files
import shutil

print("\n Copying files to organized structure...")

for img_path in tqdm(train_images, desc="Training set"):
    src_img = os.path.join(DATA_DIR, img_path)
    src_txt = src_img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')

    # Create flat filename
    flat_name = img_path.replace('/', '_')
    dst_img = os.path.join(f'{ORGANIZED_DIR}/train', flat_name)
    dst_txt = dst_img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')

    shutil.copy2(src_img, dst_img)
    if os.path.exists(src_txt):
        shutil.copy2(src_txt, dst_txt)
    else:
        # Create basic caption from filename
        with open(dst_txt, 'w') as f:
            category = img_path.split('/')[0]
            f.write(f"{category} artwork, art, artistic")

for img_path in tqdm(val_images, desc="Validation set"):
    src_img = os.path.join(DATA_DIR, img_path)
    src_txt = src_img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')

    flat_name = img_path.replace('/', '_')
    dst_img = os.path.join(f'{ORGANIZED_DIR}/val', flat_name)
    dst_txt = dst_img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')

    shutil.copy2(src_img, dst_img)
    if os.path.exists(src_txt):
        shutil.copy2(src_txt, dst_txt)
    else:
        with open(dst_txt, 'w') as f:
            category = img_path.split('/')[0]
            f.write(f"{category} artwork, art, artistic")

print(" Dataset organized!")

### Find and Remove Corrupted Images
Run this before training to clean your dataset

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def check_and_clean_dataset(data_dir, remove_corrupted=True):
    """
    Check all images and remove corrupted ones

    Args:
        data_dir: Directory containing images
        remove_corrupted: If True, delete corrupted files
    """
    print(f" Checking images in: {data_dir}")

    # Get all image files
    image_files = [
        f for f in os.listdir(data_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
    ]

    print(f" Found {len(image_files)} images to check")

    corrupted_files = []

    # Check each image
    for img_file in tqdm(image_files, desc="Checking images"):
        img_path = os.path.join(data_dir, img_file)

        try:
            # Try to open and verify
            with Image.open(img_path) as img:
                img.verify()  # Verify it's a valid image

            # Try to actually load it
            with Image.open(img_path) as img:
                img.load()  # Force load the image data
                img.convert('RGB')  # Convert to RGB

        except Exception as e:
            print(f"\n Corrupted: {img_file}")
            print(f"   Error: {e}")
            corrupted_files.append(img_path)

    # Report results
    print(f"\n{'='*60}")
    print(f"RESULTS")
    print(f"{'='*60}")
    print(f" Valid images: {len(image_files) - len(corrupted_files)}")
    print(f" Corrupted images: {len(corrupted_files)}")

    if corrupted_files:
        print(f"\n Corrupted files:")
        for f in corrupted_files:
            print(f"   - {os.path.basename(f)}")

        if remove_corrupted:
            print(f"\n  Removing corrupted files...")
            for f in corrupted_files:
                try:
                    os.remove(f)
                    # Also remove caption file if exists
                    txt_file = f.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
                    if os.path.exists(txt_file):
                        os.remove(txt_file)
                    print(f"   Removed: {os.path.basename(f)}")
                except Exception as e:
                    print(f"   Error removing {f}: {e}")

            print(f"\n Cleanup complete!")
        else:
            print(f"\n  Set remove_corrupted=True to delete these files")
    else:
        print(f"\n All images are valid!")

    return len(image_files) - len(corrupted_files), len(corrupted_files)


if __name__ == '__main__':
    # Check both train and val directories
    dirs_to_check = [
        './data/art_organized/train',
        './data/art_organized/val',
        './data/art_images_organized/train',
        './data/art_images_organized/val'
    ]

    for data_dir in dirs_to_check:
        if os.path.exists(data_dir):
            print(f"\n{'='*60}")
            print(f"Checking: {data_dir}")
            print(f"{'='*60}")

            valid, corrupted = check_and_clean_dataset(
                data_dir,
                remove_corrupted=True  # Set to True to auto-remove
            )

    print(f"\n{'='*60}")
    print(f" DATASET CLEANING COMPLETE")
    print(f"{'='*60}")
    print(f"\nYou can now run training without image errors!")

In [ ]:
# STEP 4: SETUP MODEL

print("\n" + "=" * 80)
print("STEP 4: INITIALIZING ATTENTION GAN MODEL")
print("=" * 80)

from attention_gan import AttentionTextToImageGAN, weights_init
from text_embedding import TextEmbedder

# Create model
print("\n  Building Attention GAN...")
model = AttentionTextToImageGAN(
    latent_dim=100,
    text_embedding_dim=768,
    ngf=64,  # Generator filters
    ndf=64,  # Discriminator filters
    num_channels=3,
    image_size=64,
    num_attention_blocks=2,
    attention_heads=8
)

# Initialize weights
model.generator.apply(weights_init)
model.discriminator.apply(weights_init)

print(" Model created!")

# Model statistics
gen_params = sum(p.numel() for p in model.generator.parameters())
disc_params = sum(p.numel() for p in model.discriminator.parameters())
total_params = gen_params + disc_params

print(f"\n Model Statistics:")
print(f"   Generator parameters: {gen_params:,}")
print(f"   Discriminator parameters: {disc_params:,}")
print(f"   Total parameters: {total_params:,}")

# Setup text embedder
print("\n Loading CLIP text embedder...")
embedder = TextEmbedder(
    model_name='openai/clip-vit-base-patch32',
    device=None
)

In [ ]:
# CREATE TRAINER

print("\n" + "=" * 80)
print("STEP 5: INITIALIZING TRAINER")
print("=" * 80)

from attention_trainer import AttentionGANTrainer

trainer = AttentionGANTrainer(
    model=model,
    text_embedder=embedder,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    learning_rate=1e-4,  # Lower LR for art (better quality)
    beta1=0.5,
    beta2=0.999,
    lambda_gp=10.0,
    n_critic=5,
    checkpoint_dir='./outputs/art_gan/checkpoints',
    log_dir='./outputs/art_gan/logs'
)

print(" Trainer initialized!")

In [ ]:
# PREPARE DATALOADER

print("\n" + "=" * 80)
print("STEP 6: PREPARING DATA LOADER")
print("=" * 80)

from torch.utils.data import DataLoader
import sys
sys.path.append('./models')
from fine_tune_stable_diffusion import CustomImageDataset

# Training dataset
train_dataset = CustomImageDataset(
    images_dir=f'{ORGANIZED_DIR}/train',
    size=64,
    tokenizer=embedder.tokenizer,
    max_length=77
)

# Validation dataset
val_dataset = CustomImageDataset(
    images_dir=f'{ORGANIZED_DIR}/val',
    size=64,
    tokenizer=embedder.tokenizer,
    max_length=77
)

# Determine batch size based on available memory
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if gpu_memory >= 16:
        batch_size = 32
    elif gpu_memory >= 8:
        batch_size = 16
    else:
        batch_size = 8
else:
    batch_size = 4

print(f"\n Batch size: {batch_size}")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")

In [ ]:
# TRAINING CONFIGURATION

print("\n" + "=" * 80)
print("STEP 7: TRAINING CONFIGURATION")
print("=" * 80)

# Training parameters
NUM_EPOCHS = 200  # Adjust based on your needs
SAVE_INTERVAL = 10
SAMPLE_INTERVAL = 5

# Validation prompts for sampling
validation_prompts = [
    "a beautiful oil painting of a landscape",
    "a detailed drawing of a portrait",
    "a sculpture of a human figure",
    "an engraving of a classical scene",
    "a painting of flowers in a vase",
    "an abstract artistic composition"
]

print(f"\n  Training Configuration:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {batch_size}")
print(f"   Learning rate: 1e-4")
print(f"   Save interval: {SAVE_INTERVAL} epochs")
print(f"   Sample interval: {SAMPLE_INTERVAL} epochs")
print(f"   Device: {trainer.device}")

print(f"\n Validation prompts:")
for i, prompt in enumerate(validation_prompts, 1):
    print(f"   {i}. {prompt}")

In [ ]:
"""
COMPLETE FIXED TRAINING SCRIPT
Dimension issue solved - Ready to run!
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPTextModel, CLIPTokenizer
from PIL import Image
import numpy as np
import os
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_DIR = './data/art_images_organized/train'
OUTPUT_DIR = './outputs/art_gan'
BATCH_SIZE = 16
NUM_EPOCHS = 150
LEARNING_RATE = 1e-4
IMAGE_SIZE = 64

# CRITICAL FIX: Use 512-dim to match CLIP output!
TEXT_EMBEDDING_DIM = 512  # ← This is the fix!

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")

os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/samples', exist_ok=True)

# ============================================================================
# TEXT EMBEDDER (FIXED DIMENSIONS!)
# ============================================================================

class FixedCLIPEmbedder:
    """
    CLIP embedder that returns correct 512 dimensions
    """

    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f" Loading CLIP (returns 512-dim)...")

        self.tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
        self.model = CLIPTextModel.from_pretrained('openai/clip-vit-base-patch32')
        self.model.to(self.device)
        self.model.eval()

        # Actual dimension from CLIP pooler_output
        self.embedding_dim = 512

        print(f" CLIP loaded (512-dim embeddings)")

    def encode_to_tensor(self, texts, normalize=True):
        if isinstance(texts, str):
            texts = [texts]

        with torch.no_grad():
            encoded = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=77,
                return_tensors='pt'
            )

            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            outputs = self.model(**encoded)

            # Use pooler_output (512-dim) - this is what CLIP actually returns!
            embeddings = outputs.pooler_output

            if normalize:
                embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings

    def get_embedding_dim(self):
        return self.embedding_dim

# ============================================================================
# DATASET
# ============================================================================

class SimpleImageTextDataset(Dataset):
    def __init__(self, images_dir, size=64, tokenizer=None):
        self.images_dir = images_dir
        self.size = size
        self.tokenizer = tokenizer

        self.image_files = [
            f for f in os.listdir(images_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        print(f" Found {len(self.image_files)} images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_file = self.image_files[idx]

        # Load image
        img_path = os.path.join(self.images_dir, img_file)
        image = Image.open(img_path).convert('RGB')
        image = image.resize((self.size, self.size), Image.LANCZOS)
        image = np.array(image).astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5
        image = torch.from_numpy(image).permute(2, 0, 1)

        # Load caption
        txt_file = img_file.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
        txt_path = os.path.join(self.images_dir, txt_file)

        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as f:
                caption = f.read().strip()
        else:
            caption = "artwork art"

        return {
            'image': image,
            'text': caption
        }

print("\n Loading model with FIXED dimensions...")

import sys
sys.path.append('./models')
from attention_gan import AttentionTextToImageGAN, weights_init

# Create model with 512-dim text embeddings (matches CLIP!)
model = AttentionTextToImageGAN(
    latent_dim=100,
    text_embedding_dim=512,  # ← FIXED! Was 768, now 512 to match CLIP
    ngf=64,
    ndf=64,
    num_channels=3,
    image_size=IMAGE_SIZE,
    num_attention_blocks=2,
    attention_heads=8
)

model.generator.apply(weights_init)
model.discriminator.apply(weights_init)
model = model.to(device)

gen_params = sum(p.numel() for p in model.generator.parameters())
disc_params = sum(p.numel() for p in model.discriminator.parameters())

print(f" Model loaded (text_embedding_dim={TEXT_EMBEDDING_DIM})")
print(f"   Generator: {gen_params:,} params")
print(f"   Discriminator: {disc_params:,} params")

# ============================================================================
# INITIALIZE EMBEDDER
# ============================================================================

embedder = FixedCLIPEmbedder()

# Verify dimensions match!
test_emb = embedder.encode_to_tensor(["test"])
print(f"\n✓ Embedder test: {test_emb.shape} (should be [1, 512])")

if test_emb.shape[1] != TEXT_EMBEDDING_DIM:
    print(f" ERROR: Dimension mismatch!")
    print(f"   Expected: {TEXT_EMBEDDING_DIM}")
    print(f"   Got: {test_emb.shape[1]}")
    exit(1)
else:
    print(f" Dimensions match perfectly!")

# ============================================================================
# PREPARE DATA
# ============================================================================

print("\n Preparing dataset...")

dataset = SimpleImageTextDataset(
    images_dir=DATA_DIR,
    size=IMAGE_SIZE,
    tokenizer=embedder.tokenizer
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

print(f" Dataset ready: {len(dataset)} images, {len(dataloader)} batches")

# ============================================================================
# TRAINING SETUP
# ============================================================================

print("\n  Setting up training...")

optimizer_G = optim.Adam(model.generator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
optimizer_D = optim.Adam(model.discriminator.parameters(), lr=LEARNING_RATE * 2, betas=(0.5, 0.999))

criterion = nn.BCELoss()

print(" Training ready")

# ============================================================================
# TRAINING LOOP
# ============================================================================

print(f"\n Starting training for {NUM_EPOCHS} epochs...")
print(f" TIP: Press Ctrl+C to stop and save checkpoint\n")

try:
    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"{'='*60}")

        model.generator.train()
        model.discriminator.train()

        epoch_g_loss = 0
        epoch_d_loss = 0
        num_g_updates = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")

        for batch_idx, batch in enumerate(pbar):
            real_images = batch['image'].to(device)
            texts = batch['text']

            batch_size = real_images.size(0)

            # Get text embeddings
            with torch.no_grad():
                text_embeddings = embedder.encode_to_tensor(texts, normalize=True)

            # Labels
            real_labels = torch.ones(batch_size, 1, device=device)
            fake_labels = torch.zeros(batch_size, 1, device=device)

            # ============= Train Discriminator =============
            optimizer_D.zero_grad()

            real_validity = model.discriminator(real_images, text_embeddings)
            d_real_loss = criterion(real_validity, real_labels)

            noise = torch.randn(batch_size, model.latent_dim, device=device)
            fake_images = model.generator(noise, text_embeddings)
            fake_validity = model.discriminator(fake_images.detach(), text_embeddings)
            d_fake_loss = criterion(fake_validity, fake_labels)

            d_loss = (d_real_loss + d_fake_loss) / 2
            d_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.discriminator.parameters(), 1.0)
            optimizer_D.step()

            epoch_d_loss += d_loss.item()

            # ============= Train Generator =============
            if batch_idx % 5 == 0:
                optimizer_G.zero_grad()

                noise = torch.randn(batch_size, model.latent_dim, device=device)
                fake_images = model.generator(noise, text_embeddings)
                fake_validity = model.discriminator(fake_images, text_embeddings)

                g_loss = criterion(fake_validity, real_labels)
                g_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.generator.parameters(), 1.0)
                optimizer_G.step()

                epoch_g_loss += g_loss.item()
                num_g_updates += 1

                pbar.set_postfix({
                    'G': f"{g_loss.item():.4f}",
                    'D': f"{d_loss.item():.4f}"
                })
            else:
                pbar.set_postfix({
                    'D': f"{d_loss.item():.4f}"
                })

        # Epoch summary
        avg_g_loss = epoch_g_loss / max(num_g_updates, 1)
        avg_d_loss = epoch_d_loss / len(dataloader)

        print(f"\n Epoch {epoch+1} Summary:")
        print(f"   Generator Loss: {avg_g_loss:.4f}")
        print(f"   Discriminator Loss: {avg_d_loss:.4f}")

        # Save checkpoint
        if (epoch + 1) % 10 == 0 or (epoch + 1) == NUM_EPOCHS:
            checkpoint_path = f'{OUTPUT_DIR}/checkpoints/model_epoch_{epoch+1}.pt'
            torch.save({
                'epoch': epoch + 1,
                'generator': model.generator.state_dict(),
                'discriminator': model.discriminator.state_dict(),
                'optimizer_G': optimizer_G.state_dict(),
                'optimizer_D': optimizer_D.state_dict(),
                'g_loss': avg_g_loss,
                'd_loss': avg_d_loss,
            }, checkpoint_path)
            print(f" Checkpoint saved: {checkpoint_path}")

        # Generate samples
        if (epoch + 1) % 5 == 0 or (epoch + 1) == NUM_EPOCHS:
            print(f" Generating samples...")
            model.generator.eval()

            test_prompts = [
                "a beautiful painting of a landscape",
                "a detailed drawing of a portrait",
                "a sculpture of a human figure"
            ]

            with torch.no_grad():
                test_emb = embedder.encode_to_tensor(test_prompts)
                samples = model.generate(test_emb, num_samples=1)

            for i, img in enumerate(samples):
                img_tensor = (img + 1) / 2
                img_tensor = img_tensor.clamp(0, 1)
                img_array = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')
                Image.fromarray(img_array).save(
                    f'{OUTPUT_DIR}/samples/epoch{epoch+1}_sample{i}.png'
                )

            print(f"   Saved {len(test_prompts)} samples")

except KeyboardInterrupt:
    print("\n\n  Training interrupted!")
    checkpoint_path = f'{OUTPUT_DIR}/checkpoints/model_interrupted.pt'
    torch.save({
        'epoch': epoch + 1,
        'generator': model.generator.state_dict(),
        'discriminator': model.discriminator.state_dict(),
    }, checkpoint_path)
    print(f"💾 Saved: {checkpoint_path}")

print("\n TRAINING COMPLETE!")
print(f" Checkpoints: {OUTPUT_DIR}/checkpoints/")
print(f" Samples: {OUTPUT_DIR}/samples/")